# 🛒 PROJETO SUPERMERCADO — Sistema Completo
**Grupo:** Alisson Silva, Ana Luiza Lima, Julio Cesar Brito, João Pedro Ceo, Mateus Dantas e Levi Ramos

---
## Parte 1 — Salvar e Carregar o Modelo Treinado

Este notebook treina o modelo Apriori, salva em arquivo `.pkl` com **joblib** e confirma
que o modelo recarregado gera exatamente as mesmas previsões que o original.

In [ ]:
!pip install mlxtend joblib -q

import pandas as pd
import numpy as np
import joblib
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print('✅ Bibliotecas carregadas!')

### 1) Treinar o modelo (mesmo das atividades anteriores)

In [ ]:
dados = {
    'ID_Compra': [1,2,3,4,5,6,7,8,9,10],
    'Nome_Produto': [
        'Arroz, Feijão, Óleo',  'Pão, Leite, Café',
        'Arroz, Feijão',        'Arroz, Carne',
        'Arroz, Feijão, Carne', 'Pão, Leite',
        'Arroz, Óleo',          'Feijão, Carne, Óleo',
        'Arroz, Feijão, Leite', 'Pão, Café, Leite',
    ],
}
df = pd.DataFrame(dados)

transacoes = [[p.strip() for p in r.split(',')] for r in df['Nome_Produto']]
te = TransactionEncoder()
df_bin = pd.DataFrame(te.fit_transform(transacoes), columns=te.columns_)
produtos = list(te.columns_)

MIN_SUPPORT, MIN_CONFIDENCE = 0.30, 0.60
itemsets = apriori(df_bin, min_support=MIN_SUPPORT, use_colnames=True)
regras = association_rules(itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)

print(f'✅ Modelo treinado — {len(regras)} regras geradas')
regras[['antecedents','consequents','support','confidence','lift']]

### 2) Função de previsão
Recebe os produtos do carrinho e retorna as recomendações com a confiança (probabilidade).

In [ ]:
def prever(carrinho, regras_df, top_n=3):
    """Aplica as regras de associação e retorna recomendações ordenadas pela confiança."""
    entrada = frozenset(p.strip().capitalize() for p in carrinho)
    recs = {}
    for _, r in regras_df.iterrows():
        if r['antecedents'].issubset(entrada):
            for prod in r['consequents'] - entrada:
                if prod not in recs or r['confidence'] > recs[prod]['confianca']:
                    recs[prod] = {
                        'produto':   prod,
                        'confianca': float(r['confidence']),
                        'lift':      float(r['lift']),
                    }
    resultado = sorted(recs.values(), key=lambda x: x['confianca'], reverse=True)
    return resultado[:top_n]

# Teste rápido
print('Teste — carrinho [Arroz]:')
for r in prever(['Arroz'], regras):
    print(f"  → {r['produto']}: {r['confianca']:.0%} de confiança (lift {r['lift']:.2f})")

### 3) Salvar o modelo com joblib (.pkl)

> **Sobre o scaler:** o enunciado pede para salvar o `StandardScaler`/`MinMaxScaler` **quando aplicável**.
> No nosso projeto os dados já são **binários (0 = não comprou / 1 = comprou)**, então **não usamos normalização**
> e **não há scaler para salvar**. Salvamos apenas o modelo (as regras) e os metadados necessários.

In [ ]:
# Empacotamos tudo que a interface precisa em um único objeto
modelo_completo = {
    'regras':         regras,          # as regras de associação treinadas
    'produtos':       produtos,        # lista de produtos conhecidos
    'min_support':    MIN_SUPPORT,     # parâmetros usados no treino
    'min_confidence': MIN_CONFIDENCE,
    'scaler':         None,            # não aplicável (dados binários)
}

# Salva em arquivo .pkl
joblib.dump(modelo_completo, 'modelo_supermercado.pkl')
print('✅ Modelo salvo em: modelo_supermercado.pkl')

import os
tamanho = os.path.getsize('modelo_supermercado.pkl')
print(f'   Tamanho do arquivo: {tamanho} bytes')

### 4) Carregar o modelo e VERIFICAR que as previsões são idênticas (obrigatório)

In [ ]:
# Recarrega o modelo do arquivo
modelo_carregado = joblib.load('modelo_supermercado.pkl')
regras_carregadas = modelo_carregado['regras']
print('✅ Modelo recarregado do arquivo .pkl\n')

# Compara as previsões do modelo ORIGINAL vs RECARREGADO
carrinhos_teste = [['Arroz'], ['Pão'], ['Feijão'], ['Leite'], ['Café']]

print('VERIFICAÇÃO — Modelo Original  ×  Modelo Recarregado')
print('=' * 60)
tudo_igual = True
for carrinho in carrinhos_teste:
    orig = prever(carrinho, regras)
    novo = prever(carrinho, regras_carregadas)
    igual = orig == novo
    tudo_igual = tudo_igual and igual
    status = '✅ IDÊNTICO' if igual else '❌ DIFERENTE'
    rec_orig = ', '.join(f"{r['produto']}({r['confianca']:.0%})" for r in orig) or 'nenhuma'
    print(f'  Carrinho {str(carrinho):<12} → {rec_orig:<30} {status}')

print('=' * 60)
if tudo_igual:
    print('✅ CONFIRMADO: o modelo recarregado gera EXATAMENTE as mesmas previsões.')
else:
    print('❌ ATENÇÃO: houve diferença entre os modelos.')

### 5) Baixar o arquivo .pkl (para usar na interface Streamlit)
Execute a célula abaixo no Google Colab para baixar o modelo salvo.

In [ ]:
# Apenas no Google Colab — baixa o arquivo .pkl para o seu computador
try:
    from google.colab import files
    files.download('modelo_supermercado.pkl')
    print('✅ Download iniciado!')
except ImportError:
    print('Fora do Colab — o arquivo modelo_supermercado.pkl está salvo na pasta atual.')